Create Patient Labs Embeddings per Visit

In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM (GB):", torch.cuda.get_device_properties(0).total_memory / 1e9)


Torch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA A10-24Q
VRAM (GB): 25.769345024


In [2]:
# =========================================================
# LABEVENT → MedGemma Embeddings (per admission)
# Target cohort: Solid Cancer Patients (ICD-9 / ICD-10)
# =========================================================
import os
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from collections import defaultdict

# ------------------------------
# CONFIG
# ------------------------------
LABEVENTS_PATH = r".....mimic iv\mimic-iv-3.1\hosp\labevents.csv.gz"
DLABITEMS_PATH = r"......mimic iv\mimic-iv-3.1\hosp\d_labitems.csv.gz"
DIAGNOSES_PATH = r"......mimic iv\mimic-iv-3.1\hosp\diagnoses_icd.csv.gz"

OUTPUT_PATH = r"....solid_cancer_lab_embs.npy"
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

MODEL_NAME = "google/medgemma-4b-pt"
MAX_LEN = 512
MAX_LABS_PER_ADMISSION = 200  # avoid extreme truncation bias

# ------------------------------
# TARGET ICD CODES (SOLID CANCERS)
# ------------------------------
TARGET_ICD9_PREFIXES = (
    "140", "141", "142", "143", "144", "145", "146", "147", "148", "149",
    "153", "154",
    "162",
    "174",
    "185",
    "188"
)

TARGET_ICD10_PREFIXES = (
    "C00", "C01", "C02", "C03", "C04", "C05", "C06", "C07", "C08",
    "C18", "C19", "C20",
    "C34",
    "C50",
    "C61",
    "C67"
)

# ------------------------------
# DEVICE & MODEL
# ------------------------------
print("Loading MedGemma...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModel.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    dtype=torch.bfloat16
)
model.eval()
first_device = next(model.parameters()).device
print("Model loaded.\n")

# ------------------------------
# EMBEDDING FUNCTION
# ------------------------------
@torch.inference_mode()
def embed_text(text: str) -> np.ndarray:
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    )
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    outputs = model(**inputs)
    mask = inputs["attention_mask"].unsqueeze(-1)
    hidden = outputs.last_hidden_state

    emb = (hidden * mask).sum(dim=1) / mask.sum(dim=1)
    return emb.float().cpu().numpy()[0]

# ------------------------------
# LOAD TARGET ADMISSIONS
# ------------------------------
print("Loading solid cancer diagnoses...")
diag = pd.read_csv(
    DIAGNOSES_PATH,
    usecols=["subject_id", "hadm_id", "icd_code", "icd_version"]
)

diag["icd_code"] = diag["icd_code"].astype(str).str.upper().str.strip()

target_mask = (
    ((diag.icd_version == 9) & diag.icd_code.str.startswith(TARGET_ICD9_PREFIXES)) |
    ((diag.icd_version == 10) & diag.icd_code.str.startswith(TARGET_ICD10_PREFIXES))
)

target_hadm_ids = set(diag.loc[target_mask, "hadm_id"].unique())
print(f"Target admissions: {len(target_hadm_ids)}\n")

# ------------------------------
# LOAD LAB EVENTS
# ------------------------------
print("Loading lab events...")
labs = pd.read_csv(
    LABEVENTS_PATH,
    usecols=["subject_id", "hadm_id", "itemid", "valuenum", "valueuom", "flag"]
)

labs = labs[labs["hadm_id"].isin(target_hadm_ids)]
labs = labs.dropna(subset=["valuenum"])

lab_items = pd.read_csv(
    DLABITEMS_PATH,
    usecols=["itemid", "label"]
)

labs = labs.merge(lab_items, on="itemid", how="left")
labs = labs.dropna(subset=["label"])

print(f"Lab rows after filtering: {len(labs)}")

# ------------------------------
# LABS → TEXT PER ADMISSION
# ------------------------------
admission_lab_text = defaultdict(list)
admission_subject = {}

for _, row in labs.iterrows():
    if len(admission_lab_text[row["hadm_id"]]) >= MAX_LABS_PER_ADMISSION:
        continue

    line = f"{row['label']}: {row['valuenum']}"
    if pd.notna(row["valueuom"]):
        line += f" {row['valueuom']}"
    if pd.notna(row["flag"]):
        line += f" ({row['flag']})"

    admission_lab_text[row["hadm_id"]].append(line)
    admission_subject[row["hadm_id"]] = row["subject_id"]

print(f"Admissions with labs: {len(admission_lab_text)}\n")

# ------------------------------
# EMBEDDINGS PER ADMISSION
# ------------------------------
lab_embeddings, hadm_ids, subject_ids = [], [], []

for i, (hid, lab_lines) in enumerate(admission_lab_text.items(), 1):
    text = (
        "Laboratory results during this hospital admission:\n"
        + ". ".join(lab_lines)
    )

    try:
        emb = embed_text(text)
    except RuntimeError as e:
        if "out of memory" in str(e):
            torch.cuda.empty_cache()
            continue
        else:
            raise e

    if not np.isnan(emb).any():
        lab_embeddings.append(emb)
        hadm_ids.append(hid)
        subject_ids.append(admission_subject[hid])

    if i % 100 == 0:
        print(f"Embedded {i}/{len(admission_lab_text)} admissions")

    torch.cuda.empty_cache()

# ------------------------------
# SAVE
# ------------------------------
lab_embeddings = np.vstack(lab_embeddings)
np.save(OUTPUT_PATH, lab_embeddings)
np.save(OUTPUT_PATH.replace(".npy", "_hadm_ids.npy"), np.array(hadm_ids))
np.save(OUTPUT_PATH.replace(".npy", "_subject_ids.npy"), np.array(subject_ids))

print("\n✅ LAB embeddings complete")
print("Embeddings shape:", lab_embeddings.shape)
print("Saved to:", OUTPUT_PATH)


Loading MedGemma...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded.

Loading solid cancer diagnoses...
Target admissions: 21769

Loading lab events...
Lab rows after filtering: 3733803
Admissions with labs: 19866

Embedded 100/19866 admissions
Embedded 200/19866 admissions
Embedded 300/19866 admissions
Embedded 400/19866 admissions
Embedded 500/19866 admissions
Embedded 600/19866 admissions
Embedded 700/19866 admissions
Embedded 800/19866 admissions
Embedded 900/19866 admissions
Embedded 1000/19866 admissions
Embedded 1100/19866 admissions
Embedded 1200/19866 admissions
Embedded 1300/19866 admissions
Embedded 1400/19866 admissions
Embedded 1500/19866 admissions
Embedded 1600/19866 admissions
Embedded 1700/19866 admissions
Embedded 1800/19866 admissions
Embedded 1900/19866 admissions
Embedded 2000/19866 admissions
Embedded 2100/19866 admissions
Embedded 2200/19866 admissions
Embedded 2300/19866 admissions
Embedded 2400/19866 admissions
Embedded 2500/19866 admissions
Embedded 2600/19866 admissions
Embedded 2700/19866 admissions
Embedded 280